In [1]:
	
import numpy as np
import csv

In [2]:
# =========================
# 1. Leitura do CSV
# =========================

def load_data(filename):
    X = []
    y = []

    with open(filename, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            # Target
            survived = row["survived"]
            if survived == "":
                continue

            y.append(int(survived))

            # Features selecionadas
            # Vamos usar:
            # pclass, sex, age, sibsp, parch, fare

            pclass = float(row["pclass"]) if row["pclass"] != "" else 0

            # Convertendo sexo
            sex = 1 if row["sex"] == "female" else 0

            age = float(row["age"]) if row["age"] != "" else -1

            # Se estiver vazio, o valor será 0
            # Se não estiver vazio, converte pra float
            sibsp = float(row["sibsp"]) if row["sibsp"] != "" else 0
            parch = float(row["parch"]) if row["parch"] != "" else 0
            fare = float(row["fare"]) if row["fare"] != "" else 0

            X.append([pclass, sex, age, sibsp, parch, fare])
        
    return np.array(X), np.array(y).reshape(-1, 1)

In [3]:
# =========================
# 2. Tratamento de Dados
# =========================

def fill_missing_age(X):
    ages = X[:, 2]
    mean_age = np.mean(ages[ages != -1])

    X[:, 2] = np.where(ages == -1, mean_age, ages)
    return X

def normalize(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - mean) / (std + 1e-8)

def add_bias(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones, X))

In [4]:
# =========================
# 3. Funções do Modelo
# =========================

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_cost(X, y, beta):
    m = len(y)
    h = sigmoid(X @ beta)
    epsilon = 1e-8

    cost = -(1/m) * np.sum(
        y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon)
    )

    return cost

def gradient_descent(X, y, beta, lr, epochs):
    m = len(y)

    for i in range(epochs):
        h = sigmoid(X @ beta)
        gradient = (1/m) * (X.T @ (h - y))

        beta = beta - lr * gradient

        if i % 100 == 0:
            print(f"Epoch {i} - Cost: {compute_cost(X, y, beta):.4f}")

    return beta

In [5]:
# =========================
# 4. Treinamento
# =========================

def train(X, y):
    X = fill_missing_age(X)
    X = normalize(X)
    X = add_bias(X)

    beta = np.zeros((X.shape[1], 1))

    beta = gradient_descent(X, y, beta, lr=0.01, epochs=2000)

    return beta, X

In [6]:
# ===========================================
# 5. Predição e Cálculo da Matriz de Confusão
# ===========================================

def predict(X, beta):
    probs = sigmoid(X @ beta)
    return (probs >= 0.5).astype(int)

def confusion_matrix(y_true, y_pred):
    # Achatar os arrays para garantir que tenham apenas 1 dimensão
    y_t = y_true.flatten()
    y_p = y_pred.flatten()

    # O operador '&' faz a comparação bit a bit (element-wise) nos arrays do numpy
    # Faz AND lógico, elemento por elemento de cada array.
    # Por exemplo:
    # y_t = np.array([1, 0, 1, 1])
    # cond1 = (y_t == 1)
    # Resultado: [True, False, True, True]
    VP = np.sum((y_t == 1) & (y_p == 1)) 
    VN = np.sum((y_t == 0) & (y_p == 0)) 
    FP = np.sum((y_t == 0) & (y_p == 1)) 
    FN = np.sum((y_t == 1) & (y_p == 0)) 

    return VP, VN, FP, FN

def calculate_metrics(VP, VN, FP, FN):
    acc = (VP + VN) / (VP + VN + FP + FN)
    prec = VP / (VP + FP)
    rec = VP / (VP + FN)
    spec = VN / (VN + FP)
    f1 = 2 * (prec * rec) / (prec + rec)

    return acc, prec, rec, spec, f1

In [7]:
# =========================
# 6. Split Treino / Teste
# =========================

def train_test_split(X, y, test_size=0.2):
    np.random.seed(42)

    indices = np.arange(len(X))
    np.random.shuffle(indices)

    split = int(len(X) * (1 - test_size))

    train_idx = indices[:split]
    test_idx = indices[split:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [8]:
# ===================================
# 7. Execução e Exibição das Métricas
# ===================================

X, y = load_data("titanic.csv")

X_train, X_test, y_train, y_test = train_test_split(X, y)

beta, X_train_processed = train(X_train, y_train)

# Processar teste com MESMAS transformações
X_test = fill_missing_age(X_test)
X_test = normalize(X_test)
X_test = add_bias(X_test)

# Vetor de Predições
y_pred = predict(X_test, beta)

# Métricas da Matriz de Confusão
VP, VN, FP, FN = confusion_matrix(y_test, y_pred)

print("\n" + "=" * 30)
print("Matriz de Confusão")
print("=" * 30)
print(f"Verdadeiros Positivos (VP): {VP}")
print(f"Verdadeiros Negativos (VN): {VN}")
print(f"Falsos Positivos (FP): {FP}")
print(f"Falsos Negativos (FN): {FN}")

# 2. Calcular e exibir as Métricas
acc, prec, rec, spec, f1 = calculate_metrics(VP, VN, FP, FN)

print("\n" + "=" * 30)
print("Métricas de Avaliação")
print("=" * 30)
print(f"Acurácia: {acc:.4f} ({(acc*100):.1f}%)")
print(f"Precisão: {prec:.4f} ({(prec*100):.1f}%)")
print(f"Revocação: {rec:.4f} ({(rec*100):.1f}%)")
print(f"Especificidade: {spec:.4f} ({(spec*100):.1f}%)")
print(f"F1-Score: {f1:.4f} ({(f1*100):.1f}%)")

Epoch 0 - Cost: 0.6918
Epoch 100 - Cost: 0.5970
Epoch 200 - Cost: 0.5460
Epoch 300 - Cost: 0.5160
Epoch 400 - Cost: 0.4970
Epoch 500 - Cost: 0.4844
Epoch 600 - Cost: 0.4757
Epoch 700 - Cost: 0.4694
Epoch 800 - Cost: 0.4648
Epoch 900 - Cost: 0.4614
Epoch 1000 - Cost: 0.4588
Epoch 1100 - Cost: 0.4568
Epoch 1200 - Cost: 0.4552
Epoch 1300 - Cost: 0.4539
Epoch 1400 - Cost: 0.4529
Epoch 1500 - Cost: 0.4521
Epoch 1600 - Cost: 0.4515
Epoch 1700 - Cost: 0.4509
Epoch 1800 - Cost: 0.4505
Epoch 1900 - Cost: 0.4501

Matriz de Confusão
Verdadeiros Positivos (VP): 68
Verdadeiros Negativos (VN): 123
Falsos Positivos (FP): 32
Falsos Negativos (FN): 39

Métricas de Avaliação
Acurácia: 0.7290 (72.9%)
Precisão: 0.6800 (68.0%)
Revocação: 0.6355 (63.6%)
Especificidade: 0.7935 (79.4%)
F1-Score: 0.6570 (65.7%)
